In [2]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(sys.prefix).parent
%cd {PROJECT_ROOT}

/home/younes/younes/Projects/Python/barid_internship


In [ ]:
import polars as pl
import altair as alt

import seaborn as sns
import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
# import statsmodels.api as sm
# from statsmodels.graphics.tsaplots import plot_acf
# import calendar
# from fpppy.utils import plot_series, plot_series_stacked, plot_diagnostics
# from scipy.stats import pearsonr
# from plotly import express as px
# from pathlib import Path
# from lxml import html
# from itertools import chain
from typing import Callable, Literal

import importlib
import lib
import offline_historique
import read_data

importlib.reload(lib)
importlib.reload(offline_historique)
from offline_historique import parse_historique_from_folder  # noqa: E402

cfg = pl.Config()
cfg.set_tbl_width_chars(10000)
cfg.set_fmt_str_lengths(100)
cfg.set_tbl_cols(-1)
cfg.set_tbl_rows(10)

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [3]:
cabs_domestic = set(read_data.read_smi_suiviexpedition_many()["CAB"])

In [4]:
fields = (
    pl.read_parquet("data/cab_dfs/fields.parquet")
    .filter(Etat="V")
    .filter(pl.col("Cab").is_in(cabs_domestic))
    .filter(pl.col("Date_depot").ge(pl.date(2023, 1, 1)))
    .cast({"Id": pl.String})
)
operations = (
    pl.read_parquet("data/cab_dfs/operations.parquet").filter(is_valid="V")
    # .sort("cab", "id", "Heure_Syst_Oper")
    .filter(pl.col("cab").is_in(cabs_domestic))
)
delivery = pl.read_parquet("data/cab_dfs/delivery.parquet").filter(
    pl.col("cab").is_in(cabs_domestic)
)
services = pl.read_parquet("data/cab_dfs/services.parquet").filter(
    pl.col("cab").is_in(cabs_domestic)
)

In [144]:
df = (
    operations.select(
        site_name=pl.col("Agence").str.replace_all(r"\s+", " ").str.strip_chars()
    )
    .unique("site_name")
    .sort(pl.col("site_name").str.reverse())
)

In [4]:
importlib.reload(read_data)
data = read_data.read_smi_envoisbyproduitintern_many()
data = data.filter(pl.date(2023, 1, 1) < pl.col("datedepot"))
print(data.columns)
print(data.describe())
print(data.select(pl.all().n_unique()))

['codeenvoi_', 'datedepot', 'origine', 'destination_', 'site_depot_', 'pouds_reel_', 'longueur_', 'largeur_', 'hauteur_', 'poids_volume_']
shape: (9, 11)
┌────────────┬───────────────┬────────────────────────────┬─────────┬──────────────┬──────────────────────────────┬─────────────┬───────────┬───────────┬───────────┬───────────────┐
│ statistic  ┆ codeenvoi_    ┆ datedepot                  ┆ origine ┆ destination_ ┆ site_depot_                  ┆ pouds_reel_ ┆ longueur_ ┆ largeur_  ┆ hauteur_  ┆ poids_volume_ │
│ ---        ┆ ---           ┆ ---                        ┆ ---     ┆ ---          ┆ ---                          ┆ ---         ┆ ---       ┆ ---       ┆ ---       ┆ ---           │
│ str        ┆ str           ┆ str                        ┆ str     ┆ str          ┆ str                          ┆ f64         ┆ f64       ┆ f64       ┆ f64       ┆ f64           │
╞════════════╪═══════════════╪════════════════════════════╪═════════╪══════════════╪══════════════════════════════╪═══

In [ ]:
import normalize
importlib.reload(normalize)

print(normalize.normalize_agence(operations))

shape: (2_077_161, 15)
┌──────────────────────┬──────────┬────────────────┬─────────────────────┬────────────────────────┬────────────────────────────────┬────────────┬──────────────────────┬───────────┬───────────┬─────────┬─────────────┬──────────┬─────────────┬───────────────────────┐
│ cab                  ┆ id       ┆ Date_operation ┆ Heure_Syst_Oper     ┆ Statut                 ┆ Agence                         ┆ Agent_oper ┆ Etat                 ┆ Date_Etat ┆ Agent_maj ┆ ORIGINE ┆ status_code ┆ is_valid ┆ Agence_city ┆ Agence_type           │
│ ---                  ┆ ---      ┆ ---            ┆ ---                 ┆ ---                    ┆ ---                            ┆ ---        ┆ ---                  ┆ ---       ┆ ---       ┆ ---     ┆ ---         ┆ ---      ┆ ---         ┆ ---                   │
│ str                  ┆ str      ┆ date           ┆ datetime[μs]        ┆ str                    ┆ str                            ┆ i64        ┆ str                  ┆ str       

In [ ]:
print(
    normalize_sites(operations, patterns)
    .filter(pl.col("pattern_name").is_null().not_())
)

shape: (547, 3)
┌────────────────────────────────────────────┬───────────────────────────────────┬───────────────────────┐
│ site_name                                  ┆ matched_text                      ┆ pattern_name          │
│ ---                                        ┆ ---                               ┆ ---                   │
│ str                                        ┆ str                               ┆ str                   │
╞════════════════════════════════════════════╪═══════════════════════════════════╪═══════════════════════╡
│ BC ZGHANGHAN (NADOR)                       ┆ ZGHANGHAN (NADOR)                 ┆ BC                    │
│ AGENCE MESSAGERIE MOHAMMEDIA 97065         ┆ MOHAMMEDIA 97065                  ┆ AGENCE MESSAGERIE     │
│ BC Casa Mohamed 6                          ┆ Casa Mohamed 6                    ┆ BC                    │
│ BC SOUK LARBAA                             ┆ SOUK LARBAA                       ┆ BC                    │
│ BC CASABLANCA AIN S

In [ ]:
print(
    df.with_columns(
        [pl.lit(1).alias("1"), pl.lit(2).alias("2")],
    )
)

shape: (1_574, 3)
┌───────────────────────────────┬─────┬─────┐
│ site_name                     ┆ 1   ┆ 2   │
│ ---                           ┆ --- ┆ --- │
│ str                           ┆ i32 ┆ i32 │
╞═══════════════════════════════╪═════╪═════╡
│ BC ZGHANGHAN (NADOR)          ┆ 1   ┆ 2   │
│ SETTAT SAMIR-                 ┆ 1   ┆ 2   │
│ MARRAKECH BASE DES E.A.       ┆ 1   ┆ 2   │
│ Site inexistant : 0           ┆ 1   ┆ 2   │
│ Site inexistant : 50230       ┆ 1   ┆ 2   │
│ …                             ┆ …   ┆ …   │
│ BC Fes Soundous               ┆ 1   ┆ 2   │
│ SAHATEL Demnat                ┆ 1   ┆ 2   │
│ BC CASA BOENDOG-Renault       ┆ 1   ┆ 2   │
│ BC Taroudant                  ┆ 1   ┆ 2   │
│ BC Casa Quartier des hopitaux ┆ 1   ┆ 2   │
└───────────────────────────────┴─────┴─────┘
